In [112]:
import openml
from openml import config
config.apikey = 'c0d6200b271e73a8aec0904980876c3c'
import pandas as pd 
import numpy as np
from sklearn.impute import KNNImputer

### Load the benchmark suite

In [123]:
suite_num = 99
# suite_num = 225
benchmark_suite = openml.study.get_suite(suite_num)


print(f'Benchmark Suite: {benchmark_suite.name}')
print(f'Number of datasets in benchmark suite: {len(benchmark_suite.tasks)}')


dataset_ids = {openml.tasks.get_task(task_id).dataset_id for task_id in benchmark_suite.tasks}


quality_list = openml.datasets.list_qualities()
print(f'A full set of meta-features would be: {len(quality_list)}')


Benchmark Suite: OpenML-CC18 Curated Classification benchmark
Number of datasets in benchmark suite: 72
A full set of meta-features would be: 107


### Creates raw Dataframe with datasets (rows) and meta-features (columns)

In [124]:
rows = []

for dataset_id in sorted(dataset_ids):
    try:
        dataset = openml.datasets.get_dataset(dataset_id, download_qualities=True)
        qualities = dataset.qualities

        missing_qualities = set(quality_list) - set(qualities.keys())
        if missing_qualities:
            print(f"Dataset {dataset_id} is missing {len(missing_qualities)} meta-features")

        row = {"dataset_id": dataset_id}
        for q in quality_list:
            row[q] = qualities.get(q, None)

        rows.append(row)

    except Exception as e:
        print(f"Failed to load dataset {dataset_id}: {e}")

print(f"\nTotal datasets collected: {len(rows)}")
df = pd.DataFrame(rows)
df.to_csv(f"{suite_num}_qualities_raw.csv", index=False)
print(f'Unprocessed csv with qualites saved to {suite_num}_qualities_raw.csv')

Dataset 40499 is missing 45 meta-features
Dataset 40668 is missing 88 meta-features
Dataset 40670 is missing 45 meta-features
Dataset 40701 is missing 45 meta-features
Dataset 40923 is missing 45 meta-features
Dataset 40927 is missing 45 meta-features
Dataset 40966 is missing 45 meta-features
Dataset 40975 is missing 45 meta-features
Dataset 40978 is missing 45 meta-features
Dataset 40979 is missing 45 meta-features
Dataset 40982 is missing 45 meta-features
Dataset 40983 is missing 45 meta-features
Dataset 40984 is missing 45 meta-features
Dataset 40994 is missing 45 meta-features
Dataset 40996 is missing 45 meta-features
Dataset 41027 is missing 45 meta-features

Total datasets collected: 72
Unprocessed csv with qualites saved to 99_qualities_raw.csv


In [126]:
features_only = df.drop(columns=["dataset_id"])

print(f"\nDataFrame Analysis:")
print(f"Shape: {features_only.shape}")
print(f"Missing values before imputation: {features_only.isnull().sum().sum()}")



DataFrame Analysis:
Shape: (72, 107)
Missing values before imputation: 1619


In [127]:
missing_cols = features_only.columns[features_only.isnull().any()].tolist()
print(f"Columns with missing values: {len(missing_cols)}")


completely_empty_cols = features_only.columns[features_only.isnull().all()].tolist()
print(f"Completely empty columns: {len(completely_empty_cols)}")
print(f"Completely empty column names: {completely_empty_cols[:10]}")

Columns with missing values: 88
Completely empty columns: 0
Completely empty column names: []...


In [137]:
features_cleaned = features_only.drop(columns=completely_empty_cols)
print(f"\nShape after dropping empty columns: {features_cleaned.shape}")


Shape after dropping empty columns: (72, 107)


In [138]:
total_features = len(features_cleaned.columns)
total_cells = features_cleaned.size
missing_cells = features_cleaned.isnull().sum().sum()

print(f"Total features: {total_features}")
print(f"Total data points: {total_cells}")
print(f"Missing data points: {missing_cells} ({missing_cells / total_cells * 100:.1f}%)")

Total features: 107
Total data points: 7704
Missing data points: 1619 (21.0%)


In [139]:
missing_percentage = (features_cleaned.isnull().sum() / len(features_cleaned)) * 100

print(f"Features with 0% missing: {(missing_percentage == 0).sum()}")
print(f"Features with 1–25% missing: {((missing_percentage > 0) & (missing_percentage <= 25)).sum()}")
print(f"Features with 26–50% missing: {((missing_percentage > 25) & (missing_percentage <= 50)).sum()}")
print(f"Features with 51–75% missing: {((missing_percentage > 50) & (missing_percentage <= 75)).sum()}")
print(f"Features with 76–99% missing: {((missing_percentage > 75) & (missing_percentage < 100)).sum()}")
print(f"Features with 100% missing: {(missing_percentage == 100).sum()}")


Features with 0% missing: 19
Features with 1–25% missing: 74
Features with 26–50% missing: 0
Features with 51–75% missing: 14
Features with 76–99% missing: 0
Features with 100% missing: 0


In [140]:
worst_missing = missing_percentage.sort_values(ascending=False)
for feature, pct in worst_missing.head(10).items():
    print(f"{feature}: {pct:.1f}% missing")

MeanNoiseToSignalRatio: 70.8% missing
EquivalentNumberOfAtts: 70.8% missing
Quartile1AttributeEntropy: 69.4% missing
Quartile3MutualInformation: 69.4% missing
Quartile3AttributeEntropy: 69.4% missing
Quartile2MutualInformation: 69.4% missing
Quartile2AttributeEntropy: 69.4% missing
MaxAttributeEntropy: 69.4% missing
MeanMutualInformation: 69.4% missing
MaxMutualInformation: 69.4% missing


In [144]:
missing_per_dataset = features_cleaned.isnull().sum(axis=1)

# Get the index (i.e., row number) of the max and min
idx_max = missing_per_dataset.idxmax()
idx_min = missing_per_dataset.idxmin()

# Look up the corresponding dataset_id
dataset_id_max = df.loc[idx_max, "dataset_id"]
dataset_id_min = df.loc[idx_min, "dataset_id"]

# Print with dataset IDs
print(f"Dataset with the most missing features: ID {dataset_id_max} - {missing_per_dataset[idx_max]} missing")
print(f"Dataset with the least missing features: ID {dataset_id_min} - {missing_per_dataset[idx_min]} missing")

Dataset with the most missing features: ID 40668 - 88 missing
Dataset with the least missing features: ID 23 - 0 missing


In [145]:
reliable_features = missing_percentage[missing_percentage < 10].index.tolist()
problematic_features = missing_percentage[missing_percentage > 50].index.tolist()

print(f"Highly reliable features (<10% missing): {len(reliable_features)}")
print(f"Problematic features (>50% missing): {len(problematic_features)}")

print(f"\nMissing data per dataset distribution:")
print(f"  Min: {missing_per_dataset.min()} features missing")
print(f"  25th percentile: {missing_per_dataset.quantile(0.25):.1f}")
print(f"  Median: {missing_per_dataset.median():.1f}")
print(f"  75th percentile: {missing_per_dataset.quantile(0.75):.1f}")
print(f"  Max: {missing_per_dataset.max()}")


Highly reliable features (<10% missing): 24
Problematic features (>50% missing): 14

Missing data per dataset distribution:
  Min: 0 features missing
  25th percentile: 14.0
  Median: 14.0
  75th percentile: 24.0
  Max: 88


In [ ]:
imputer = KNNImputer(n_neighbors=5)
features_imputed = imputer.fit_transform(features_cleaned)

df_imputed = pd.DataFrame(features_imputed, columns=features_cleaned.columns)
df_imputed.insert(0, "dataset_id", df["dataset_id"])

print(f"Missing values after imputation: {df_imputed.isnull().sum().sum()}")
print(f"Final dataset shape: {df_imputed.shape}")

In [ ]:
csv_filename = f'{suite_num}_qualities_imputed.csv'
df_imputed.to_csv(csv_filename, index=False)
print(f"Saved imputed dataset as: {csv_filename}")

### Example dataset breakdown - qualities available and unavailable

In [ ]:
def get_unavailable_qualities(dataset_id, quality_list):
    print(f"\nAnalyzing meta-features for dataset {dataset_id}...")

    try:
        dataset = openml.datasets.get_dataset(dataset_id, download_qualities=True)
        qualities = dataset.qualities

        missing_qualities = []
        nan_qualities = []
        valid_qualities = []

        for qual_name in quality_list:
            value = qualities.get(qual_name, None)
            if value is None:
                missing_qualities.append(qual_name)
            elif pd.isna(value):
                nan_qualities.append(qual_name)
            else:
                valid_qualities.append(qual_name)

        print(f"  Valid qualities     : {len(valid_qualities)}")
        print(f"  NaN qualities       : {len(nan_qualities)}")
        print(f"  Missing qualities   : {len(missing_qualities)}")
        print(f"  Total defined in list: {len(quality_list)}")

        return {
            "valid_qualities": valid_qualities,
            "nan_qualities": nan_qualities,
            "missing_qualities": missing_qualities
        }

    except Exception as e:
        print(f"  Error loading dataset {dataset_id}: {e}")
        return {
            "valid_qualities": [],
            "nan_qualities": [],
            "missing_qualities": [],
            "error": str(e)
        }

In [ ]:
def get_quality_value(dataset_id, quality_name):
    try:
        dataset = openml.datasets.get_dataset(dataset_id, download_qualities=True)
        value = dataset.qualities.get(quality_name, None)
        return value
    except Exception as e:
        print(f"Error retrieving quality for dataset {dataset_id}: {e}")
        return None


In [136]:
print(get_quality_value(40978,'EquivalentNumberOfAtts'))

122.58576196805384


In [135]:
for id in dataset_ids:
    val = get_quality_value(id, 'EquivalentNumberOfAtts')
    if not pd.isna(val):
        print(f"{id} → detected")

3 → detected
40975 → detected
40978 → detected
23 → detected
151 → detected
29 → detected
31 → detected
38 → detected
46 → detected
50 → detected
1461 → detected
1590 → detected
4534 → detected
188 → detected
6332 → detected
1480 → detected
1486 → detected
469 → detected
23381 → detected
40670 → detected
40701 → detected


In [ ]:
result = get_unavailable_qualities(dataset_id=1466, quality_list=quality_list)